In [ ]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker('en_IN')

# CONFIG
total_records = 322_465
unique_reviews_count = 50000
chunk_size = 322465
start_date = datetime(2024, 5, 1)
end_date = datetime(2025, 4, 30)
date_range = (end_date - start_date).days

# Fixed pools
cities = [
    "Chennai", "Mumbai", "Delhi", "Bangalore", "Hyderabad", "Ahmedabad", "Pune", "Kolkata",
    "Coimbatore", "Madurai", "Tiruchirappalli", "Salem", "Erode", "Tirunelveli", "Vellore", "Thoothukudi",
    "Nagercoil", "Thanjavur", "Cuddalore", "Dindigul", "Kanchipuram", "Nagapattinam", "Karur", "Sivakasi",
    "Pudukkottai", "Hosur", "Ambur", "Jaipur", "Lucknow", "Patna", "Surat", "Varanasi", "Visakhapatnam",
    "Jodhpur", "Guwahati", "Dehradun", "Chandigarh", "Thrissur", "Warangal", "Guntur", "Noida"
]

brands = [
    "Swiggy", "Zomato", "Domino's", "KFC", "McDonald's", "Burger King", "Subway",
    "Pizza Hut", "Haldiram's", "Cafe Coffee Day"
]

products = [
    "Masala Dosa", "Paneer Butter Masala", "Chicken Biryani", "Veg Pulao", "Tandoori Chicken",
    "Margherita Pizza", "Cheeseburger", "Veggie Wrap", "Chole Bhature", "Fish Curry", "Idli Sambar",
    "Pav Bhaji", "Chicken Shawarma", "Hakka Noodles", "Fried Rice", "Spring Roll", "Gulab Jamun",
    "Ice Cream Sundae", "Kathi Roll", "Aloo Paratha"
]

cuisines = ['Indian', 'Chinese', 'South Indian', 'North Indian', 'Italian', 'Mexican', 'Thai', 'Fast Food']
health_prefs = ['Vegetarian', 'Vegan', 'Gluten-Free', 'None']
price_sensitivity = ['Low', 'Medium', 'High']
num_customers = 125_521

# Generate customer IDs and gender
customer_ids = [f"CUST{str(i).zfill(6)}" for i in range(1, num_customers + 1)]
other_gender_ids = set(random.sample(range(1, num_customers + 1), 50))
genders = ['M', 'F']

# Generate 50,000 unique human-like reviews with matching ratings
base_phrases = [
    ("Very tasty and fresh. Great portion.", 5),
    ("Good taste but slightly over-salted.", 4),
    ("Burger was cold and dry. Not worth it.", 2),
    ("Absolutely loved the spicy flavors and generous serving.", 5),
    ("Presentation was nice but the dish was a bit bland.", 3),
    ("Great value for money, will order again.", 5),
    ("Not satisfied, too oily and small portion.", 2),
    ("Dessert was delightful and not overly sweet.", 5),
    ("Service was slow but the food made up for it.", 4),
    ("Unique flavors I haven’t tried before, recommended!", 5)
]

def generate_human_review():
    starters = ["Delicious", "Pretty average", "Terrible", "Loved", "Hated", "Impressed with", "Disappointed in"]
    mains = ["portion size", "flavor", "presentation", "texture", "spice level", "freshness"]
    closers = ["Would definitely order again.", "Not worth it.", "Highly recommended.", "Will avoid next time.", "Might try once more."]
    return f"{random.choice(starters)} {random.choice(mains)}. {random.choice(closers)}"

review_pool = set()
while len(review_pool) < unique_reviews_count:
    review_pool.add(generate_human_review())

review_pool = list(review_pool)
ratings_distribution = [5]*50 + [4]*25 + [3]*15 + [2]*7 + [1]*3
review_rating_map = {review: random.choice(ratings_distribution) for review in review_pool}

# Generate and save in chunks
record_id = 1
file_index = 1

while record_id <= total_records:
    data = []
    end_id = min(record_id + chunk_size - 1, total_records)

    for i in range(record_id, end_id + 1):
        cust_idx = random.randint(0, num_customers - 1)
        customer_id = customer_ids[cust_idx]
        gender = 'Other' if (cust_idx + 1) in other_gender_ids else random.choice(genders)
        age = random.randint(18, 70)
        city = random.choice(cities)
        brand = random.choice(brands)
        product = random.choice(products)
        review = random.choice(review_pool)
        rating = review_rating_map[review]
        date = start_date + timedelta(days=random.randint(0, date_range))
        cuisine = random.choice(cuisines)
        health = random.choices(health_prefs, weights=[35, 10, 10, 45])[0]
        price = random.choice(price_sensitivity)
        recommend = "Yes" if rating >= 4 else "No"

        data.append([
            i, customer_id, age, gender, city, brand, product,
            rating, review, date.date(), cuisine, health, price, recommend
        ])

    df = pd.DataFrame(data, columns=[
        "ReviewID", "CustomerID", "Age", "Gender", "City", "Brand", "Product",
        "Rating", "Review", "Date", "Cuisine", "HealthPref", "PriceSense", "Recommend"
    ])

    filename = f"Food_Review_Dataset_Part_{file_index}.xlsx"
    df.to_excel(filename, index=False)
    print(f"Saved: {filename}")

    file_index += 1
    record_id = end_id + 1
